# Lesson 3: Claude Tool Calling In LangChain

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

# Default model is Haiku 4.5; override with e.g. LLM_MODEL=claude-opus-4-8
MODEL = os.environ.get("LLM_MODEL", "claude-haiku-4-5")

# Pydantic Syntax

In [2]:
class User:
    def __init__(self,name:str,age:int,email:str):
        self.name=name
        self.age=age
        self.email=email
    

In [3]:
foo=User(name="Joe",age=32,email="joe@gmail.com")

In [4]:
foo.name

'Joe'

In [5]:
foo=User(name="Joe",age="bar",email="joe@gmail.com")
foo.age

'bar'

In [6]:
from typing import List
from pydantic import BaseModel, Field

class pUser(BaseModel):
    name:str
    age:int
    email:str

In [7]:
foo_p=pUser(name="Joe",age=32,email="joe@gmail.com")
foo_p.name

'Joe'

In [8]:
foo_p=pUser(name="Joe",age="55",email="joe@gmail.com")
foo_p.age

55

In [9]:
class Class(BaseModel):
    students:List[pUser]

In [10]:
obj=Class(students=[pUser(name="Joe",age=32,email="joe@gmail")])

obj

Class(students=[pUser(name='Joe', age=32, email='joe@gmail')])

# Pydantic to tool definition

In [11]:
class WeatherSearch(BaseModel):
    """Call this with an airport code to get the weather at that airport"""
    airport_code: str = Field(description="airport code to get weather for")

In [ ]:
# convert_to_openai_tool is a provider-agnostic LangChain helper that turns a
# Pydantic model into a JSON-Schema tool definition (the same shape ChatAnthropic
# uses under the hood when you call .bind_tools).
from langchain_core.utils.function_calling import convert_to_openai_tool

weather_tool = convert_to_openai_tool(WeatherSearch)

weather_tool

In [13]:
class WeatherSearch2(BaseModel):
    """Call this with an airport code to get the weather at that airport"""
    airport_code: str

In [ ]:
convert_to_openai_tool(WeatherSearch2)

In [ ]:
from langchain_anthropic import ChatAnthropic

model = ChatAnthropic(model=MODEL)

In [ ]:
# Pass the Pydantic model straight to .bind_tools — no manual conversion needed
model.bind_tools([WeatherSearch]).invoke("what is the weather in SF today?")

In [ ]:
model_with_function = model.bind_tools([WeatherSearch])

In [18]:
model_with_function.invoke("what is the weather in sf")

AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"airport_code":"SFO"}', 'name': 'WeatherSearch'}, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 68, 'total_tokens': 85, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-C2iTbgxalwUYrrFvILGIjD7mOmn5c', 'service_tier': 'default', 'finish_reason': 'function_call', 'logprobs': None}, id='run--566810a1-28ca-404a-a0a5-d040ba2d1ad1-0', usage_metadata={'input_tokens': 68, 'output_tokens': 17, 'total_tokens': 85, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

# Forcing it to use a function

In [ ]:
# Force the model to call a specific tool by name
model_with_forced_function = model.bind_tools([WeatherSearch], tool_choice="WeatherSearch")

In [20]:
model_with_forced_function.invoke("what is the weather in sf?")

AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"airport_code":"SFO"}', 'name': 'WeatherSearch'}, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 79, 'total_tokens': 86, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-C2iTgC2XdGT4dEWsue9EuSpWlnjV7', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--bd21537d-55e0-4fd6-b087-10b52e2e15c1-0', usage_metadata={'input_tokens': 79, 'output_tokens': 7, 'total_tokens': 86, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [21]:
model_with_forced_function.invoke("hi!")

AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"airport_code":"JFK"}', 'name': 'WeatherSearch'}, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 7, 'prompt_tokens': 74, 'total_tokens': 81, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-C2iTj1CIPAmM4WaPikafki9pDjf92', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--7eae35d0-475b-4d5e-9ef8-385b67d03bb0-0', usage_metadata={'input_tokens': 74, 'output_tokens': 7, 'total_tokens': 81, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

# Using in a chain

In [22]:
from langchain.prompts import ChatPromptTemplate

prompt=ChatPromptTemplate.from_messages([("system","You are a helpful assistant"),
("user","{input}")])

chain=prompt|model_with_function
chain.invoke({"input": "what is the weather in sf?"})

AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"airport_code":"SFO"}', 'name': 'WeatherSearch'}, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 75, 'total_tokens': 92, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-C2iTnzImusrjXtGgCBMJPw8GLyaHP', 'service_tier': 'default', 'finish_reason': 'function_call', 'logprobs': None}, id='run--d0224766-d3ae-4ea0-b574-b0ea70dcde6a-0', usage_metadata={'input_tokens': 75, 'output_tokens': 17, 'total_tokens': 92, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

# Using multiple functions

In [23]:
class ArtistSearch(BaseModel):
    """Call this to get the names of songs by a particular artist"""
    artist_name: str = Field(description="name of artist to look up")
    n: int = Field(description="number of results")

In [ ]:
tools = [WeatherSearch, ArtistSearch]

In [ ]:
model_with_functions = model.bind_tools(tools)
model_with_functions.invoke("what is the weather in sf?")

In [26]:
model_with_functions.invoke("what are three songs by taylor swift?")

AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"artist_name":"taylor swift","n":3}', 'name': 'ArtistSearch'}, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 118, 'total_tokens': 140, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-C2iU3vxKD7FTyjgy2e51whOLNy339', 'service_tier': 'default', 'finish_reason': 'function_call', 'logprobs': None}, id='run--9edd3f51-9bba-4528-8cda-5e41e64cbe80-0', usage_metadata={'input_tokens': 118, 'output_tokens': 22, 'total_tokens': 140, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [27]:
model_with_functions.invoke("hi!")

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 111, 'total_tokens': 121, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-C2iU4nioCIPENFtijAtou2AKNOWAJ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--4885010b-e3d2-4a53-a263-681d97c86c4a-0', usage_metadata={'input_tokens': 111, 'output_tokens': 10, 'total_tokens': 121, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})